## Task 5: Stacked Area Chart of Cumulative Installs
**Author:** Vansh Sharma


- Cleaned the Google Play Store dataset and removed duplicate apps.
- Applied filters based on rating, reviews, app size, category, and app name.
- Translated selected category names for visualization.
- Calculated cumulative installs by month for each category.
- Identified months with more than 25% month-over-month install growth.
- Created a stacked area chart with highlighted high-growth periods.
- Configured the visualization to display only between **4:00 PM IST and 6:00 PM IST**.
### Key Insights:
- Total Apps: **2,137**
- Categories: **6**
- Cumulative Installs: **22.90 Billion**
- Top Category: **TOOLS**
- High Growth Months: **7**


In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from zoneinfo import ZoneInfo
import pytz

In [ ]:
play_store_data = pd.read_csv(
    r"C:\Users\Vansh Sharma\Downloads\Play Store Data (1).csv"
)
play_store_data=pd.DataFrame(play_store_data)


In [46]:
play_store_data.head(2)

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19.0,10000,Free,0,Everyone,Art & Design,2018-01-07,1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0,Everyone,Art & Design;Pretend Play,2018-01-15,2.0.0,4.0.3 and up


## Data Cleaning

- Removed corrupted rows from the dataset.

- Converted **Installs** and **Reviews** columns into integer format.

- Converted **Last Updated** column into datetime format.

- Handled missing values in Rating, Version, Type, and Size columns.

- Converted app size values from KB/MB format into numerical MB values.

- Filled missing Size values using the median value.

- Removed duplicate app records to ensure data consistency.

In [7]:
# Remove corrupted row
play_store_data = play_store_data[
    play_store_data["Installs"] != "Free"
]

# Convert Installs into integer
play_store_data["Installs"] = (
    play_store_data["Installs"]
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
    .astype(int)
)

# Convert Reviews into integer
play_store_data["Reviews"] = (
    play_store_data["Reviews"]
    .astype(int)
)

# Convert Last Updated into datetime
play_store_data["Last Updated"] = pd.to_datetime(
    play_store_data["Last Updated"],
    errors="coerce"
)

# Fill missing values
play_store_data["Rating"] = (
    play_store_data["Rating"]
    .fillna(play_store_data["Rating"].median())
)

play_store_data["Current Ver"] = (
    play_store_data["Current Ver"].ffill()
)

play_store_data["Android Ver"] = (
    play_store_data["Android Ver"].ffill()
)

play_store_data["Type"] = (
    play_store_data["Type"]
    .fillna(play_store_data["Type"].mode()[0])
)

# Convert Size into MB
play_store_data["Size"] = (
    play_store_data["Size"]
    .replace("Varies with device", np.nan)
)

# Convert KB to MB
play_store_data.loc[
    play_store_data["Size"].str.contains("k", na=False),
    "Size"
] = (
    play_store_data.loc[
        play_store_data["Size"].str.contains("k", na=False),
        "Size"
    ]
    .str.replace("k", "", regex=False)
    .astype(float)
    / 1024
)

# Convert MB values
play_store_data.loc[
    play_store_data["Size"].str.contains("M", na=False),
    "Size"
] = (
    play_store_data.loc[
        play_store_data["Size"].str.contains("M", na=False),
        "Size"
    ]
    .str.replace("M", "", regex=False)
    .astype(float)
)

# Convert Size to float
play_store_data["Size"] = pd.to_numeric(
    play_store_data["Size"],
    errors="coerce"
)

# Fill missing Size values
play_store_data["Size"] = (
    play_store_data["Size"]
    .fillna(play_store_data["Size"].median())
)

# Remove duplicate apps
play_store_data = play_store_data.drop_duplicates(
    subset="App",
    keep="first"
)

# Check cleaned dataset
play_store_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9659 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   App             9659 non-null   object        
 1   Category        9659 non-null   object        
 2   Rating          9659 non-null   float64       
 3   Reviews         9659 non-null   int64         
 4   Size            9659 non-null   float64       
 5   Installs        9659 non-null   int64         
 6   Type            9659 non-null   object        
 7   Price           9659 non-null   object        
 8   Content Rating  9659 non-null   object        
 9   Genres          9659 non-null   object        
 10  Last Updated    9659 non-null   datetime64[ns]
 11  Current Ver     9659 non-null   object        
 12  Android Ver     9659 non-null   object        
dtypes: datetime64[ns](1), float64(2), int64(2), object(8)
memory usage: 1.0+ MB


## Data Filtering

- Applied filtering conditions to focus on high-quality applications.

- Included only apps with:
  - Average rating of **4.2 or higher**
  - More than **1,000 reviews**
  - App size between **20 MB and 80 MB**
  - App names without numerical characters
  - Categories starting with **T** or **P**

- Created a filtered dataset containing relevant apps for further time-based install analysis and visualization.

- Filtered dataset was prepared for monthly trend analysis and category-wise comparison.

In [11]:
filtered = play_store_data[
    (play_store_data["Rating"] >= 4.2) &
    (play_store_data["Reviews"] > 1000) &
    (play_store_data["Size"].between(20, 80)) &
    (~play_store_data["App"].str.contains(r"\d", regex=True, na=False)) &
    (play_store_data["Category"].str.startswith(("T", "P")))
]

filtered.shape

(110, 13)

## Monthly Install Analysis

- Created a **Month-Year** column from the app update date for time-based analysis.

- Aggregated total installs for each app category on a monthly basis.

- Prepared category-wise monthly install data for cumulative analysis and stacked area chart visualization.

In [23]:
# Create Month-Year column
filtered["Month"] = (
    filtered["Last Updated"]
    .dt.to_period("M")
    .astype(str)
)

# Monthly installs by category
monthly_installs = (
    filtered
    .groupby(
        ["Month", "Category"],
        as_index=False
    )["Installs"]
    .sum()
)

monthly_installs.head()

C:\Users\Vansh Sharma\AppData\Local\Temp\ipykernel_21276\2938098413.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered["Month"] = (


,Month,Category,Installs
0,2014-11,PHOTOGRAPHY,1000000
1,2016-10,PRODUCTIVITY,1000000
2,2016-12,PERSONALIZATION,1000000
3,2017-03,PHOTOGRAPHY,50000000
4,2017-06,PHOTOGRAPHY,10000000


## Category Translation

- Created a **Category Display** column to keep original category names unchanged while applying translations for visualization.

- Translated selected categories for the chart legend:
  - Travel & Local → French
  - Productivity → Spanish
  - Photography → Japanese

- Prepared translated category labels for better visualization and presentation.

In [27]:
translation = {
    "TRAVEL_AND_LOCAL": "Voyage et Local",
    "PRODUCTIVITY": "Productividad",
    "PHOTOGRAPHY": "写真"
}


filtered_data["Category_Display"] = (
    filtered_data["Category"]
    .replace(translation)
)


filtered_data[
    ["Category","Category_Display"]
].drop_duplicates()

,Category,Category_Display
2801,PHOTOGRAPHY,写真
3102,TRAVEL_AND_LOCAL,Voyage et Local
3233,TOOLS,TOOLS
3352,PERSONALIZATION,PERSONALIZATION
3450,PRODUCTIVITY,Productividad
3575,PARENTING,PARENTING


# Pivot Tabel
- Created a **Pivot Table** to summarize monthly installs across different app categories.

- Structured the pivot table with:
  - **Rows:** Month
  - **Columns:** App Category
  - **Values:** Total Installs

- Arranged months in calendar order and filled missing values with zero for accurate time-series analysis.

- Prepared the summarized data for cumulative install calculation and stacked area chart visualization.

In [41]:
# Create Pivot Table
install_pivot = (
    filtered_data
    .pivot_table(
        index="Month",                 # Rows
        columns="Category_Display",    # Columns (App Category)
        values="Installs",             # Values
        aggfunc="sum",                 # Total Installs
        fill_value=0
    )
)

# Arrange months in calendar order
month_order = [
    "Jan","Feb","Mar","Apr","May","Jun",
    "Jul","Aug","Sep","Oct","Nov","Dec"
]

install_pivot = (
    install_pivot
    .reindex(month_order)
    .fillna(0)
)

display(install_pivot)

Category_Display,PARENTING,PERSONALIZATION,Productividad,TOOLS,Voyage et Local,写真
Month,,,,,,
Jan,110010,28276500,115781015,113815120,61672100,63014300
Feb,0,14641810,7373150,101237045,26000,239523130
Mar,1310000,16659350,36822600,100484461,6291510,201310000
Apr,70000,11824831,192167110,134050120,15244105,93630010
May,18225000,219351150,586330130,123033895,4653110,119222100
Jun,5220100,110664150,211965905,711286225,122182215,321303600
Jul,3105000,474617743,1408484523,2892714458,1298939350,1217007355
Aug,3270000,573223377,3143921265,3448929015,1370664701,2101491100
Sep,0,12221250,11357250,21414200,150,19216800


# Cumulative_installs
- Calculated **cumulative installs** using cumulative sum to analyze the overall growth trend of app categories over time.

- Calculated **Month-over-Month (MoM) growth percentage** to measure changes in cumulative installs.

- Identified high-growth periods where any app category showed more than **25% month-over-month growth**.

- Counted the number of months with significant growth for highlighting in the final visualization.

In [45]:
cumulative_installs = area_data.cumsum()


growth = (
    cumulative_installs
    .pct_change()
    .mul(100)
)
# Check where any category has more than 25% MoM growth

high_growth_months = growth.gt(25).any(axis=1)

high_growth_months.sum()

np.int64(7)

# Growth 
- Extracted the periods where install growth exceeded the **25% month-over-month threshold**.

- Filtered the growth data to identify significant growth months across different app categories.

- Used these high-growth periods for highlighting important growth trends in the stacked area chart.

In [38]:
growth_periods = growth[
    high_growth_months
]

display(growth_periods)

Category_Display,PARENTING,PERSONALIZATION,Productividad,TOOLS,Voyage et Local,写真
Month,,,,,,
Feb,0.000000,51.780843,6.368186,88.948678,0.042158,380.109166
Mar,1190.800836,38.816417,29.899598,46.725622,10.197251,66.540527
Apr,4.929543,19.847760,120.121888,42.483220,22.421227,18.583008
May,1223.146153,307.203778,166.503004,27.365997,5.590415,19.954243
Jun,26.477795,38.061140,22.586231,124.215960,139.022220,44.831004
Jul,12.452321,118.235353,122.430082,225.305630,618.339261,117.245055
Aug,11.661866,65.433804,122.861044,82.577010,90.832146,93.192076


## Month Extraction

- Extracted the month name from the **Last Updated** date column for monthly trend analysis.

- Created a separate **Month** column to group and analyze app installs over time.

- Prepared the dataset for month-wise category performance analysis and visualization.

In [42]:
filtered_data["Month"] = (
    filtered_data["Last Updated"]
    .dt.strftime("%b")
)

filtered_data[["Last Updated", "Month"]].head()

,Last Updated,Month
2801,2018-08-06,Aug
2802,2018-08-01,Aug
2803,2018-08-02,Aug
2804,2018-01-31,Jan
2805,2018-06-27,Jun


## Visualization

- Created a **Stacked Area Chart** to visualize cumulative installs over time for each app category.

- Represented each category as a separate color band to compare growth contribution.

- Highlighted periods with more than **25% month-over-month install growth**.

- Added IST time-based control to display the chart only between **4 PM and 6 PM**.

- Improved dashboard functionality by restricting visualization visibility outside the specified time window.

In [39]:
# Current IST Time
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist).time()

# Allowed Time
start_time = datetime.strptime("16:00", "%H:%M").time()
end_time = datetime.strptime("18:00", "%H:%M").time()

if start_time <= current_time <= end_time:

    plt.figure(figsize=(14,7))

    # Stacked Area Chart
    plt.stackplot(
        area_data.index,
        area_data.T,
        labels=area_data.columns,
        alpha=0.8
    )

    # Highlight months with >25% MoM growth
    high_growth = growth.gt(25).any(axis=1)

    for month in area_data.index[high_growth]:
        plt.axvspan(
            month,
            month,
            color="yellow",
            alpha=0.35
        )

    plt.title(
        "Cumulative Installs by App Category Over Time"
    )

    plt.xlabel("Month")
    plt.ylabel("Cumulative Installs")

    plt.legend(
        title="Category",
        loc="upper left"
    )

    plt.grid(True)

    plt.tight_layout()

    plt.show()

else:

    print(
        "Graph is available only between 4 PM IST and 6 PM IST"
    )

Graph is available only between 4 PM IST and 6 PM IST


# `KPIs`

In [40]:
# KPIs

total_apps = filtered_data["App"].nunique()

total_categories = filtered_data["Category_Display"].nunique()

total_installs = cumulative_installs.iloc[-1].sum()

average_rating = round(
    filtered_data["Rating"].mean(),
    2
)

top_category = (
    filtered_data
    .groupby("Category_Display")["Installs"]
    .sum()
    .idxmax()
)

high_growth_count = (
    high_growth_months
    .sum()
)


print("Total Apps:", total_apps)
print("Total Categories:", total_categories)
print("Total Cumulative Installs:", int(total_installs))
print("Average Rating:", average_rating)
print("Top Category by Installs:", top_category)
print("High Growth Months (>25%):", high_growth_count)

Total Apps: 2137
Total Categories: 6
Total Cumulative Installs: 22902913977
Average Rating: 4.16
Top Category by Installs: TOOLS
High Growth Months (>25%): 7


## Business Insights

- The analysis covered **2,137 apps** across **6 different categories** after applying quality filters.

- The dataset achieved a total cumulative install count of approximately **22.90 Billion**, showing strong user adoption across categories.

- The average rating of analyzed apps was **4.16**, indicating positive user satisfaction.

- **TOOLS** category generated the highest number of installs among all analyzed categories.

- A total of **7 months** showed more than **25% month-over-month growth**, highlighting significant periods of user growth.